# Generative Sequential Recommendation — Colab 训练

**运行前**：Runtime → Change runtime type → **GPU**（T4 或 A100）

**需要手动上传的文件只有一个**：
- `embedding/semantic_ids.npy`（本地项目 embedding/ 目录下，~300 KB）

`data/beauty_data.pkl` 已在 GitHub 仓库中，clone 后自动存在。

---

## 运行顺序
1. **准备环境**（Cell 1–4）：GPU 检查 → 安装依赖 → Clone 仓库 → 上传文件
2. **训练生成式模型**（Cell 5–6）：train.py → evaluate.py
3. **消融实验**（Cell 7–8）：随机 ID 训练 → 对比结果（核心实验）
4. **SASRec 基线**（Cell 9，可选）：sasrec_train.py
5. **下载结果**（Cell 10）

In [ ]:
# Cell 1：确认 GPU 可用
import torch
print(f'GPU 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 型号: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  未检测到 GPU，请先到 Runtime → Change runtime type → GPU')

In [ ]:
# Cell 2：安装依赖
!pip install transformers scikit-learn -q
print('依赖安装完成 ✅')

In [ ]:
# Cell 3：克隆仓库
!git clone https://github.com/rhine-yangrui/Generative-Sequential-Recommendation.git
%cd Generative-Sequential-Recommendation
!echo '当前目录：' && pwd
!echo '仓库文件：' && ls

In [ ]:
# Cell 4：上传 semantic_ids.npy
# 运行后点击「Choose Files」，选择本地 embedding/semantic_ids.npy
import os
from google.colab import files

os.makedirs('embedding', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

print('请上传 semantic_ids.npy（本地项目的 embedding/ 文件夹里）')
uploaded = files.upload()
for fname in uploaded:
    dest = f'embedding/{fname}'
    os.rename(fname, dest)
    size_kb = os.path.getsize(dest) / 1024
    print(f'✅ 已移动到 {dest}  ({size_kb:.0f} KB)')

# 验证文件
import numpy as np
sids = np.load('embedding/semantic_ids.npy', allow_pickle=True).item()
vals = list(sids.values())
print(f'\n✅ semantic_ids.npy 加载成功：{len(sids)} items')
print(f'   示例：item {list(sids.keys())[0]} → {vals[0]}')
c1_vals = set(v[0] for v in vals)
c2_vals = set(v[1] for v in vals)
print(f'   c1 范围: {min(c1_vals)}~{max(c1_vals)}，c2 范围: {min(c2_vals)}~{max(c2_vals)}')

# 确认零冲突
unique_ids = len(set(tuple(v) for v in vals))
print(f'   唯一 Semantic ID 数: {unique_ids} / {len(sids)}  {"✓ 零冲突" if unique_ids == len(sids) else "✗ 有冲突！"}')

---
## Part 1：训练生成式推荐模型（LLM Semantic ID + GPT-2）

In [ ]:
# Cell 5：训练生成式模型
# 预计时间：T4 约 1-2 小时，A100 约 20-30 分钟
# 模型自动保存至 checkpoints/best_model.pt
!python train.py

In [ ]:
# Cell 6：评估生成式模型（all-rank Recall@K / NDCG@K）
!python evaluate.py

---
## Part 2：消融实验 — 随机 ID vs LLM Semantic ID

这是整个项目最核心的实验。用随机分配的 (c1,c2,c3) 替换语义 ID，
其他一切不变（同样的模型、训练流程、评估协议）。

如果 LLM Semantic ID 的 Recall@10 显著高于随机 ID，
就证明了**语义结构**而非自回归生成本身是性能提升的来源。

In [ ]:
# Cell 7：生成随机 Semantic ID，训练消融模型
import os, random, pickle
import numpy as np

# 备份原始 semantic_ids
os.rename('embedding/semantic_ids.npy', 'embedding/semantic_ids_llm.npy')

# 生成随机 ID（与 K_LEVELS=[4,64,256] 一致）
data = pickle.load(open('data/beauty_data.pkl', 'rb'))
random.seed(42)
random_ids = {
    iid: (random.randint(0, 3), random.randint(0, 63), random.randint(0, 255))
    for iid in data['item2id'].values()
}
np.save('embedding/semantic_ids.npy', random_ids)

# 检查冲突率（随机 ID 会有大量冲突，这是预期的）
unique_random = len(set(tuple(v) for v in random_ids.values()))
print(f'随机 ID：{len(random_ids)} items，唯一 ID 数 {unique_random}')
print(f'冲突率: {(len(random_ids) - unique_random) / len(random_ids) * 100:.1f}%')
print()

# 训练消融模型（结果保存到 checkpoints/best_model.pt，会覆盖原有的）
print('开始训练消融模型（随机 ID）...')
!python train.py

# 保存消融模型，避免被后续覆盖
import shutil
shutil.copy('checkpoints/best_model.pt', 'checkpoints/random_id_model.pt')
print('\n✅ 消融模型已保存至 checkpoints/random_id_model.pt')

In [ ]:
# Cell 8：对比结果
import os, math, pickle, random
import numpy as np
import torch
from tqdm import tqdm
from model.generative_rec import build_model
from model.inference import build_reverse_index, predict_topk

device = 'cuda' if torch.cuda.is_available() else 'cpu'
data   = pickle.load(open('data/beauty_data.pkl', 'rb'))

K_LIST     = [5, 10]
BEAM_WIDTH = 50

def quick_eval(model, semantic_ids, test_seqs, k_list=K_LIST, beam_width=BEAM_WIDTH):
    sid_to_item, sid_array, item_id_list = build_reverse_index(semantic_ids)
    model.eval()
    metrics = {k: {'Recall': [], 'NDCG': []} for k in k_list}
    for user, full_seq in tqdm(test_seqs.items(), desc='Evaluating', leave=False):
        target  = full_seq[-1]
        history = full_seq[:-1]
        recs    = predict_topk(model, history, semantic_ids, sid_to_item,
                               sid_array, item_id_list,
                               k=max(k_list), beam_width=beam_width, device=device)
        for k in k_list:
            topk = recs[:k]
            if target in topk:
                rank = topk.index(target) + 1
                metrics[k]['Recall'].append(1)
                metrics[k]['NDCG'].append(1 / math.log2(rank + 1))
            else:
                metrics[k]['Recall'].append(0)
                metrics[k]['NDCG'].append(0.0)
    return {k: {m: np.mean(v) for m, v in mv.items()} for k, mv in metrics.items()}

# 评估 LLM Semantic ID 模型
print('加载 LLM Semantic ID 模型...')
llm_ids   = np.load('embedding/semantic_ids_llm.npy', allow_pickle=True).item()
llm_model = build_model().to(device)
# 注意：best_model.pt 现在是消融模型，LLM 模型需要单独保存（若已保存则加载）
# 如果 Cell 5/6 之后就备份了 LLM 模型，请先备份：
# shutil.copy('checkpoints/best_model.pt', 'checkpoints/llm_id_model.pt')
if os.path.exists('checkpoints/llm_id_model.pt'):
    llm_model.load_state_dict(torch.load('checkpoints/llm_id_model.pt', map_location=device, weights_only=True))
    print('✅ 加载 checkpoints/llm_id_model.pt')
else:
    print('⚠️  未找到 llm_id_model.pt，请在训练完 LLM 模型后手动备份：')
    print('    shutil.copy("checkpoints/best_model.pt", "checkpoints/llm_id_model.pt")')

# 评估随机 ID 模型
print('\n加载随机 ID 模型...')
rand_ids   = np.load('embedding/semantic_ids.npy', allow_pickle=True).item()
rand_model = build_model().to(device)
rand_model.load_state_dict(torch.load('checkpoints/random_id_model.pt', map_location=device, weights_only=True))

print('\n评估 LLM Semantic ID 模型...')
llm_result  = quick_eval(llm_model, llm_ids, data['test'])
print('评估随机 ID 模型...')
rand_result = quick_eval(rand_model, rand_ids, data['test'])

# 恢复原始 semantic_ids.npy
os.rename('embedding/semantic_ids_llm.npy', 'embedding/semantic_ids.npy')

# 打印对比表
tiger_sasrec = {5: (0.0387, 0.0249), 10: (0.0605, 0.0318)}
tiger_tiger  = {5: (0.0454, 0.0321), 10: (0.0648, 0.0384)}

print('\n' + '='*70)
print('  消融实验结果（All-rank Recall@K / NDCG@K，Amazon Beauty）')
print('='*70)
print(f'  {"Model":<30}  {"R@5":>7}  {"N@5":>7}  {"R@10":>7}  {"N@10":>7}')
print('  ' + '-'*58)

rows = [
    ('SASRec (TIGER paper)',      tiger_sasrec),
    ('TIGER (original paper)',    tiger_tiger),
    ('Generative + LLM Sem. ID', {k: (llm_result[k]['Recall'], llm_result[k]['NDCG']) for k in K_LIST}),
    ('Generative + Random ID',   {k: (rand_result[k]['Recall'], rand_result[k]['NDCG']) for k in K_LIST}),
]
for name, res in rows:
    r5, n5   = res.get(5,  (0, 0))
    r10, n10 = res.get(10, (0, 0))
    print(f'  {name:<30}  {r5:>7.4f}  {n5:>7.4f}  {r10:>7.4f}  {n10:>7.4f}')

print('='*70)
gain = llm_result[10]['Recall'] - rand_result[10]['Recall']
print(f'\n  Recall@10 提升（LLM vs Random）: +{gain:.4f}')
if gain > 0:
    print('  ✓ 语义 ID 显著优于随机 ID，证明 Semantic ID 设计有效')
else:
    print('  ⚠️  LLM ID 未超过随机 ID，检查训练是否充分')

---
## Part 3：SASRec 基线（可选，也可在本地跑）

SASRec 使用随机整数 ID embedding，无语义信息，作为判别式对比基线。
评估协议与生成式模型相同（all-rank Recall@K）。

In [ ]:
# Cell 9：训练 SASRec 基线
# 预计时间：T4 约 30-60 分钟（200 epochs，early stopping）
!python baseline/sasrec_train.py

---
## Part 4：下载结果

In [ ]:
# Cell 10：下载所有 checkpoint
import os
from google.colab import files

to_download = [
    ('checkpoints/best_model.pt',       '生成式模型（当前最优，视最后一次训练而定）'),
    ('checkpoints/llm_id_model.pt',     '生成式模型（LLM Semantic ID）'),
    ('checkpoints/random_id_model.pt',  '消融模型（随机 ID）'),
    ('checkpoints/sasrec_best.pt',      'SASRec 基线'),
]

for fpath, desc in to_download:
    if os.path.exists(fpath):
        files.download(fpath)
        print(f'✅ 下载：{fpath}  ({desc})')
    else:
        print(f'⏭️  跳过：{fpath}  (不存在)')

print('\n下载完成，将 .pt 文件放到本地项目的 checkpoints/ 文件夹后，')
print('可在本地运行 python evaluate.py 复现评估结果。')